# <font color=Blue>Conversational Memory

Conversational memory is technique of enabling AI model with coherent conversation so that, it can respond to multiple queries in a chat-like manner. 


    -It allows the model to reference earlier details and instructions, maintaining a coherent "thread" rather than treating every prompt as an isolated event. 
    
    - This system relies on a context window, which determines exactly how much history the AI can "recall" at any given moment.

In LangChain, the memory strategy we choose depends on whether we are building a linear conversation or a multi-step agent. Here is the breakdown of why we use both:

**1. RunnableWithMessageHistory (Chain-Based Memory)**

This is used for simple, linear interactions where the goal is a basic back-and-forth conversation.

* Focus: It strictly manages the Chat History (the list of Human and AI messages).
* How it works: It wraps a chain, fetches the history from a store like **InMemoryChatMessageHistory**, and injects it into the prompt before the model runs.
* Can be used as a wrapper around the retriever chains when used with RAG implementation.
* Use Case: Best for standard chatbots or FAQ assistants where the LLM just needs to remember what was said previously to answer the current question.

**2. InMemorySaver (Agent-Based Memory)**

This is used for complex, stateful workflows (typically within **create_agent** agent or **LangGraph workflows**) where an agent must track its own "thoughts" and actions.

* Focus: It manages the entire Agent State, which includes not just chat history, but also tool outputs, internal variables, and the next step in a plan.
* How it works: It uses a Checkpointer to save a snapshot of the graph at every step. This allows the agent to "pause" and "resume" exactly where it left off, even if it needs to call a tool or wait for user input.
* Use Case: Essential for agents that perform multi-step tasks (like searching the web, then summarizing, then emailing) because it ensures the agent remembers which tasks it has already completed.

The Key Difference: **RunnableWithMessageHistory** remembers what was said, while **InMemorySaver** remembers what was done and where the agent is in it's process.

### <font color=Blue>Conversational Memory With InMemoryChatMessageHistory and RunnableWithMessageHistory

# InMemoryChatMessageHistory

- InMemoryChatMessageHistory is an in-memory implementation of the chat message history class within the LangChain core library.
- It stores conversation messages as a simple list in memory, allowing a chatbot or agent to maintain context across multiple interactions.

#### RunnableWithMessageHistory
RunnableWithMessageHistory is a utility wrapper in LangChain that allows any LangChain runnable (e.g., a chain or model) to be augmented with memory capabilities. It connects a memory object like  to the processing flow, enabling the LLM to access historical context automatically.

It works by:

    1. Injecting past messages (from memory) into the prompt using a placeholder.
    
    2. Automatically updating the memory after each interaction.
    
    3. Maintaining session-aware conversations via a session_id.

RunnableWithMessageHistory commonly uses **InMemoryChatMessageHistory** as a transient data store, where messages are held in a Python dictionary during the application's runtime.

| Component                    | Role                                                                |
| ---------------------------- | ------------------------------------------------------------------- |
| `RunnableWithMessageHistory` | Wraps the core chain and manages memory injection and updates       |
| `session_id`                 | Identifies and separates conversations for different users/sessions |


#### Load the model Instance

In [1]:
from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model="anthropic.claude-3-sonnet-20240229-v1:0", #'cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm.invoke("Hi")


AIMessage(content='Hello!', additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': 'f1b15801-3dcd-4665-a1ce-7c7a36b2ef3f', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sun, 22 Mar 2026 05:20:57 GMT', 'content-type': 'application/json', 'content-length': '206', 'connection': 'keep-alive', 'x-amzn-requestid': 'f1b15801-3dcd-4665-a1ce-7c7a36b2ef3f'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [316]}, 'model_provider': 'bedrock_converse', 'model_name': 'anthropic.claude-3-sonnet-20240229-v1:0'}, id='lc_run--019d13fd-3f55-7be2-b530-00172737ce62-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 5, 'total_tokens': 13, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}})

#### Create InMemoryChatMessageHistory() to store conversations

In [14]:
from rich import print

In [15]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


# Dictionary to store session-based message histories
chat_history = {}

# Function to retrieve or create chat history for a given session ID
def get_chat_history(session_id: str) -> InMemoryChatMessageHistory:
    chat_history = chats_by_session_id.get(session_id)
    if chat_history is None:
        chat_history = InMemoryChatMessageHistory()
        chats_by_session_id[session_id] = chat_history
    return chat_history


#### Create the chain with memory using the given LLM and session-aware memory callback

In [16]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Define a prompt that expects "chat_history"
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# Create the chain (combining prompt + llm)
runnable = prompt | llm

# Wrap it with history
# Note: history_messages_key must match the MessagesPlaceholder name above
chain = RunnableWithMessageHistory(
    runnable, 
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history" 
)


#### Simulate conversation with session ID "test_session"

In [17]:
# First message: Starts a conversation
response1 = chain.invoke({"input":"Hi there!"}, config={"configurable": {"session_id": "test_session"}})
print("Response 1:", response1)
print("_____"*20)

# Second message: Continues the conversation using the same session memory
response2 = chain.invoke(
    {"input":"Can you tell me the weather forecast for tomorrow in Pune?"},
    config={"configurable": {"session_id": "test_session"}}
)
print("Response 2:", response2)


Response 1:
AIMessage(
    content='Hello again!',
    additional_kwargs={},
    response_metadata={
        'ResponseMetadata': {
            'RequestId': '674b4583-ec95-470d-be35-c8c2e343be44',
            'HTTPStatusCode': 200,
            'HTTPHeaders': {
                'date': 'Sun, 22 Mar 2026 05:26:47 GMT',
                'content-type': 'application/json',
                'content-length': '215',
                'connection': 'keep-alive',
                'x-amzn-requestid': '674b4583-ec95-470d-be35-c8c2e343be44'
            },
            'RetryAttempts': 0
        },
        'stopReason': 'end_turn',
        'metrics': {'latencyMs': [480]},
        'model_provider': 'bedrock_converse',
        'model_name': 'anthropic.claude-3-sonnet-20240229-v1:0'
    },
    id='lc_run--019d1402-9454-79b2-973a-21bfcd769877-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 290,
        'output_tokens': 6,
        'total_tokens': 296,
        'input_token_details': {'cache_creation': 0, 'cache_read': 0}
    }
)

____________________________________________________________________________________________________

Response 2:
AIMessage(
    content="Unfortunately, without any additional details from you about a more specific location within the Pune 
metropolitan area, I do not have enough information to provide an accurate localized weather forecast for tomorrow 
in Pune.\n\nPune is a large city, and weather conditions can vary across different neighborhoods and suburbs. To 
give you a reliable forecast, I would need you to specify which particular area, neighborhood or nearby town you 
need the forecast for. Weather predictions are granular based on location.\n\nIf you can provide me with the name 
of the specific area, suburb or locality within or around Pune that you need the forecast for, I'd be happy to look
up the most relevant weather report for tomorrow in that place. But without that additional location context, I 
cannot narrow it down to give you a precise forecast. Please let me know if you can provide any more location 
specifics.",
    additional_kwargs={},
    response_metadata={
        'ResponseMetadata': {
            'RequestId': 'da315728-6fd9-4d77-bd41-ef9c9229bad2',
            'HTTPStatusCode': 200,
            'HTTPHeaders': {
                'date': 'Sun, 22 Mar 2026 05:26:51 GMT',
                'content-type': 'application/json',
                'content-length': '1111',
                'connection': 'keep-alive',
                'x-amzn-requestid': 'da315728-6fd9-4d77-bd41-ef9c9229bad2'
            },
            'RetryAttempts': 0
        },
        'stopReason': 'end_turn',
        'metrics': {'latencyMs': [3902]},
        'model_provider': 'bedrock_converse',
        'model_name': 'anthropic.claude-3-sonnet-20240229-v1:0'
    },
    id='lc_run--019d1402-97b7-7a93-8edf-953cca11a2b5-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 312,
        'output_tokens': 178,
        'total_tokens': 490,
        'input_token_details': {'cache_creation': 0, 'cache_read': 0}
    }
)

In [18]:
response3 = chain.invoke({"input":"What is my location?"}, config={"configurable": {"session_id": "test_session"}})
print("Response 3:", response3)


Response 3:
AIMessage(
    content="I'm afraid I do not actually know your specific location. As an AI assistant without access to data 
about your device or IP address, I have no way to definitively determine where you are located.\n\nWhen you asked 
for a weather forecast in Pune, I requested more details on the specific area or neighborhood because weather 
conditions can vary across different parts of a large metropolitan region. However, without any additional location
information provided by you, I cannot make any assumptions about which precise location in or around Pune you need 
the forecast for.\n\nUnless you are able to provide me with the name of the city, town, suburb or neighborhood you 
need the forecast for, I do not have enough contextual information to look up an accurate, localized weather 
report. I do not actually know your location since you have not shared those specifics with me. Please let me know 
if you can provide any more details about the place you need the forecast for.",
    additional_kwargs={},
    response_metadata={
        'ResponseMetadata': {
            'RequestId': '9f7a02be-397a-42e8-bacc-dd6f0d3f0357',
            'HTTPStatusCode': 200,
            'HTTPHeaders': {
                'date': 'Sun, 22 Mar 2026 05:26:55 GMT',
                'content-type': 'application/json',
                'content-length': '1178',
                'connection': 'keep-alive',
                'x-amzn-requestid': '9f7a02be-397a-42e8-bacc-dd6f0d3f0357'
            },
            'RetryAttempts': 0
        },
        'stopReason': 'end_turn',
        'metrics': {'latencyMs': [3641]},
        'model_provider': 'bedrock_converse',
        'model_name': 'anthropic.claude-3-sonnet-20240229-v1:0'
    },
    id='lc_run--019d1402-a7d1-7a52-a483-b39fcff5aa6b-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 498,
        'output_tokens': 193,
        'total_tokens': 691,
        'input_token_details': {'cache_creation': 0, 'cache_read': 0}
    }
)

In [20]:
# Access the stored chat history
chat_history = store["test_session"]

# Print each message in the history
for msg in chat_history.messages:
    print("__"*25)
    print(f"{msg.type.capitalize()}: {msg.content}")


__________________________________________________

Human: Hi there!

__________________________________________________

Ai: Hello! It's great to meet you. I'm Claude, an AI assistant created by Anthropic. I'm here to help with any 
questions or tasks you may have. Feel free to ask me about anything!

__________________________________________________

...

Ai: Unfortunately, without any additional details from you about a more specific location within the Pune 
metropolitan area, I do not have enough information to provide an accurate localized weather forecast for tomorrow 
in Pune.

Pune is a large city, and weather conditions can vary across different neighborhoods and suburbs. To give you a 
reliable forecast, I would need you to specify which particular area, neighborhood or nearby town you need the 
forecast for. Weather predictions are granular based on location.

If you can provide me with the name of the specific area, suburb or locality within or around Pune that you need 
the forecast for, I'd be happy to look up the most relevant weather report for tomorrow in that place. But without 
that additional location context, I cannot narrow it down to give you a precise forecast. Please let me know if you
can provide any more location specifics.

__________________________________________________

Human: What is my location?

__________________________________________________

Ai: I'm afraid I do not actually know your specific location. As an AI assistant without access to data about your 
device or IP address, I have no way to definitively determine where you are located.

When you asked for a weather forecast in Pune, I requested more details on the specific area or neighborhood 
because weather conditions can vary across different parts of a large metropolitan region. However, without any 
additional location information provided by you, I cannot make any assumptions about which precise location in or 
around Pune you need the forecast for.

Unless you are able to provide me with the name of the city, town, suburb or neighborhood you need the forecast 
for, I do not have enough contextual information to look up an accurate, localized weather report. I do not 
actually know your location since you have not shared those specifics with me. Please let me know if you can 
provide any more details about the place you need the forecast for.

## Refer

used in >> https://docs.langchain.com/oss/python/integrations/graphs/amazon_neptune_open_cypher?_gl=1*vg7knk*_gcl_au*Mjc3Nzk1ODA4LjE3NzA4MTAzOTY.*_ga*Mjg0ODQzMTMxLjE3NjE2Mjc0MTY.*_ga_47WX3HKKY2*czE3NzM4MTM5MzYkbzQ5JGcxJHQxNzczODEzOTQzJGo1MyRsMCRoMA..

# Conversation Window Memory 

## Sliding Window Memory

In a standard chat, the history grows linearly. Eventually, this history becomes too large for the model to "read" (exceeding the context window) or too expensive to process.

Windowing creates a boundary. Instead of sending the entire history, you only send the most relevant, recent slice. By using the last strategy, you ensure the AI always remembers the most recent turn, even if it forgets how the conversation started ten minutes ago.

Syntax & Parameter Reference
The trim_messages utility is highly configurable. Here is a breakdown of the standard syntax parameters you'll encounter:

In [16]:
#The following code demonstrates the use of ConversationBufferWindowMemory With trim_messages.

#Import the required packages

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    trim_messages,
)

#Using Amazon Bedrock ChatConverse LLM Instance

from langchain_aws import ChatBedrockConverse
llm=ChatBedrockConverse(model='cohere.command-r-plus-v1:0', #amazon.nova-lite-v1:0
                       aws_access_key_id='',
                       aws_secret_access_key='',
                       region_name='us-east-1',max_tokens=200)
llm.invoke("Hi")

#Trim messages manually like a buffer window

# External store for session memory
store = {}

# Define a buffer window message handler using trim_messages
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    # Initialize empty session memory if needed
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    # Load current messages
    messages = store[session_id].messages

    # Trim messages manually like a buffer window
    trimmed = trim_messages(
        messages=messages,
        token_counter=llm,        # You can use real token counter here if needed
        max_tokens=50,             # Buffer window size (number of messages)
        strategy="last",          # Keep most recent messages
        start_on="human",         # Ensure chat starts properly
        include_system=True,      # Keep SystemMessage if present
        allow_partial=False      # Don't allow cutting message pairs
    )

    # Re-store trimmed messages
    store[session_id] = InMemoryChatMessageHistory(messages=trimmed)

    return store[session_id]

#Build the RunnableWithMessageHistory Chain

chain = RunnableWithMessageHistory(llm, get_session_history)

# Define session ID
session_id = "window_memory_session"

#Simulate conversation turns

response1 = chain.invoke("Hi there!", config={"configurable": {"session_id": session_id}})
print("Response 1:", response1)
print("_____" * 20)

response2 = chain.invoke("Tell me a programming joke.", config={"configurable": {"session_id": session_id}})
print("Response 2:", response2)
print("_____" * 20)

response3 = chain.invoke("What is the capital of Japan?", config={"configurable": {"session_id": session_id}})
print("Response 3:", response3)
print("_____" * 20)

# Print the trimmed message history (i.e., buffer window)
print("📜 Final Trimmed Buffer Window:")
for msg in store[session_id].messages:
    print(f"{msg.type.capitalize()}: {msg.content}")

Response 1: content='Hello! How can I help you today?' additional_kwargs={} response_metadata={'ResponseMetadata': {'RequestId': 'a5405be4-c282-480c-89e6-ca75227a42ef', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 23 Mar 2026 06:57:42 GMT', 'content-type': 'application/json', 'content-length': '232', 'connection': 'keep-alive', 'x-amzn-requestid': 'a5405be4-c282-480c-89e6-ca75227a42ef'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [384]}, 'model_provider': 'bedrock_converse', 'model_name': 'cohere.command-r-plus-v1:0'} id='lc_run--019d197c-2dde-7593-8823-cfbeea69a6ec-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 3, 'output_tokens': 9, 'total_tokens': 12, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}
____________________________________________________________________________________________________
Response 2: content="Why did the programmer quit their job?\n\nBecause they didn't get arrays." additional_kwarg

In [17]:
response4 = chain.invoke("What is the capital of India?", config={"configurable": {"session_id": session_id}})
print("Response 4:", response4)
print("_____" * 20)

# Print the trimmed message history (i.e., buffer window)
print("📜 Final Trimmed Buffer Window:")
for msg in store[session_id].messages:
    print(f"{msg.type.capitalize()}: {msg.content}")

Response 4: content='The capital of India is New Delhi.' additional_kwargs={} response_metadata={'ResponseMetadata': {'RequestId': '349e1f2c-f596-4a2b-a154-6f4d5fba6c7a', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 23 Mar 2026 06:57:44 GMT', 'content-type': 'application/json', 'content-length': '235', 'connection': 'keep-alive', 'x-amzn-requestid': '349e1f2c-f596-4a2b-a154-6f4d5fba6c7a'}, 'RetryAttempts': 0}, 'stopReason': 'end_turn', 'metrics': {'latencyMs': [357]}, 'model_provider': 'bedrock_converse', 'model_name': 'cohere.command-r-plus-v1:0'} id='lc_run--019d197c-3558-7fa0-855c-cd40d05ee86c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 44, 'output_tokens': 8, 'total_tokens': 52, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}
____________________________________________________________________________________________________
📜 Final Trimmed Buffer Window:
Human: Tell me a programming joke.
Ai: Why did the programmer quit their job?
